# Day 4 Performance Analytics

This notebook computes daily returns, CAGR, Sharpe, Sortino, alpha, beta, max drawdown, composite fund scores, and benchmark tracking error from the cleaned Day 2 datasets.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPORTS_DIR = BASE_DIR / "reports"
CHART_DIR = REPORTS_DIR / "charts" / "day4"
sns.set_theme(style="whitegrid")

In [ ]:
scorecard = pd.read_csv(REPORTS_DIR / "fund_scorecard.csv", parse_dates=["drawdown_start_date", "drawdown_trough_date", "drawdown_recovery_date"])
alpha_beta = pd.read_csv(REPORTS_DIR / "alpha_beta.csv")
returns = pd.read_csv(REPORTS_DIR / "daily_returns.csv", parse_dates=["date"])
cagr = pd.read_csv(REPORTS_DIR / "cagr_comparison.csv")
tracking_error = pd.read_csv(REPORTS_DIR / "benchmark_tracking_error.csv")
drawdown = pd.read_csv(REPORTS_DIR / "max_drawdown.csv", parse_dates=["drawdown_start_date", "drawdown_trough_date", "drawdown_recovery_date"])
scorecard.head(10)

## Daily Return Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(returns["daily_return"], bins=100, kde=True)
plt.title("Distribution of Daily Fund Returns")
plt.xlabel("Daily return")
plt.show()

Daily return distribution is centered near zero with moderate tails, which is reasonable for NAV return data after calendar forward-fill.

## CAGR Comparison

In [ ]:
cagr[["scheme_name", "cagr_1yr_pct", "cagr_3yr_pct", "cagr_5yr_pct", "cagr_5yr_available"]].head(10)

The available NAV window supports full 1-year and 3-year CAGR calculations; true 5-year CAGR is marked unavailable because the cleaned NAV data starts in January 2022.

## Sharpe and Sortino Ranking

In [ ]:
scorecard[["scheme_name", "sharpe_ratio", "sortino_ratio", "score_0_100"]].head(10)

## Alpha and Beta

In [ ]:
alpha_beta.sort_values("alpha_pct", ascending=False).head(10)

Alpha and beta are estimated using OLS regression of fund daily returns against NIFTY100 daily returns.

## Maximum Drawdown

In [ ]:
scorecard[["scheme_name", "max_drawdown_pct", "drawdown_start_date", "drawdown_trough_date", "drawdown_recovery_date"]].sort_values("max_drawdown_pct").head(10)

## Fund Scorecard

In [ ]:
scorecard[[
    "scheme_name", "category", "cagr_3yr_pct", "sharpe_ratio", "alpha_pct",
    "expense_ratio_pct", "max_drawdown_pct", "score_0_100"
]].head(15)

The composite score uses:

- 30% 3-year CAGR rank
- 25% Sharpe rank
- 20% Alpha rank
- 15% inverse expense ratio rank
- 10% inverse max drawdown rank

## Benchmark Comparison

![Benchmark comparison](../reports/charts/day4/benchmark_comparison_top5_vs_indices.png)

In [ ]:
tracking_error.sort_values(["amfi_code", "benchmark"])